In [ ]:
#Benifits of Persistence
#1.Short term Memory
#2.Fault Tolerance
#3.Human in the loop
#4.Time Travel
 

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field
from typing import TypedDict, List, Annotated, Dict
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
load_dotenv()
model = ChatOpenAI()
parser = StrOutputParser()


python-dotenv could not parse statement starting at line 5


In [13]:
class Generator(TypedDict):
    topic: str
    joke: str
    explanation: str


def joke_generator(state: Generator):
    topic = state['topic']
    prompt = PromptTemplate(
        template="Generate the joke for the given input topic{topic}",
        input_variables=['topic']
    )
    chain = RunnableSequence(prompt, model, parser)
    result = chain.invoke({'topic': topic})
    return {'joke': result}


def explanation_generator(state: Generator):
    joke = state['joke']
    prompt = f"Generate the 5 line explanation for the given joke: {state['joke']}"
    result = model.invoke(prompt)
    return {'explanation': result.content}


checkpointer = InMemorySaver()  # *****

graph = StateGraph(Generator)

graph.add_node("joke_generator", joke_generator)
graph.add_node("explanation_generator", explanation_generator)
graph.add_edge(START, "joke_generator")
graph.add_edge("joke_generator", "explanation_generator")
graph.add_edge("explanation_generator", END)

workflow = graph.compile(checkpointer=checkpointer)  # ***
config1 = {'configurable': {'thread_id': "1"}}  # ***



In [14]:

initial_state = {'topic': 'pizza'}
final_state = workflow.invoke(initial_state, config=config1)  # *****
print(final_state)

{'topic': 'pizza', 'joke': 'Why did the slice of pizza go to the party? Because it wanted to be the life of the crust!', 'explanation': 'The slice of pizza went to the party because it wanted to be the center of attention. It thought it could be the life of the party by being the most popular snack. The slice believed it could create a fun and lively atmosphere with its cheesy charm. It hoped to show off its crusty personality and make everyone laugh. Ultimately, the slice just wanted to have a good time and be the star of the show.'}


In [15]:
workflow.get_state(config1)  # to ge the state of Config1 threadid

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the slice of pizza go to the party? Because it wanted to be the life of the crust!', 'explanation': 'The slice of pizza went to the party because it wanted to be the center of attention. It thought it could be the life of the party by being the most popular snack. The slice believed it could create a fun and lively atmosphere with its cheesy charm. It hoped to show off its crusty personality and make everyone laugh. Ultimately, the slice just wanted to have a good time and be the star of the show.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19ecdf-08af-6108-8002-f34dccd4a2b7'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-23T08:37:56.631783+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19ecde-f2d5-677d-8001-e353d135e490'}}, tasks=(), interrupts=())

In [16]:
list(workflow.get_state_history(config1)) # to get all intermediate state for threadid=config1

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the slice of pizza go to the party? Because it wanted to be the life of the crust!', 'explanation': 'The slice of pizza went to the party because it wanted to be the center of attention. It thought it could be the life of the party by being the most popular snack. The slice believed it could create a fun and lively atmosphere with its cheesy charm. It hoped to show off its crusty personality and make everyone laugh. Ultimately, the slice just wanted to have a good time and be the star of the show.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19ecdf-08af-6108-8002-f34dccd4a2b7'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-23T08:37:56.631783+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19ecde-f2d5-677d-8001-e353d135e490'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke

In [17]:
config2 = {'configurable': {'thread_id': "2"}}
initial_state = {'topic': 'bread'}
final_state = workflow.invoke(initial_state, config=config2)  # *****
print(final_state)

{'topic': 'bread', 'joke': 'Why did the loaf of bread go to therapy?\nBecause it had too many crumbs in its past!', 'explanation': 'The joke plays on the double meaning of the word "crumbs," referring to both literal bread crumbs and emotional baggage.\n\nThe bread going to therapy suggests that it is seeking help for some kind of issue or problem.\n\nThe punchline reveals that the bread\'s reason for seeking therapy is actually because it has "too many crumbs in its past."\n\nThis can be interpreted as a whimsical way of saying the bread has unresolved issues or regrets from its past experiences.\n\nOverall, the joke is a light-hearted play on words that uses the concept of therapy and bread crumbs to create humor.'}


In [18]:
list(workflow.get_state_history(config2)) # to get all intermediate state for threadid=config2

[StateSnapshot(values={'topic': 'bread', 'joke': 'Why did the loaf of bread go to therapy?\nBecause it had too many crumbs in its past!', 'explanation': 'The joke plays on the double meaning of the word "crumbs," referring to both literal bread crumbs and emotional baggage.\n\nThe bread going to therapy suggests that it is seeking help for some kind of issue or problem.\n\nThe punchline reveals that the bread\'s reason for seeking therapy is actually because it has "too many crumbs in its past."\n\nThis can be interpreted as a whimsical way of saying the bread has unresolved issues or regrets from its past experiences.\n\nOverall, the joke is a light-hearted play on words that uses the concept of therapy and bread crumbs to create humor.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f19ecdf-30f6-69e9-8002-73bb55a7341f'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-23T08:38:00.855396+00:00', parent_config={

In [20]:
# if we want to get the details of config1 
workflow.get_state(config1)  # since its stored in the memory we can easily fetch using threadid

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the slice of pizza go to the party? Because it wanted to be the life of the crust!', 'explanation': 'The slice of pizza went to the party because it wanted to be the center of attention. It thought it could be the life of the party by being the most popular snack. The slice believed it could create a fun and lively atmosphere with its cheesy charm. It hoped to show off its crusty personality and make everyone laugh. Ultimately, the slice just wanted to have a good time and be the star of the show.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19ecdf-08af-6108-8002-f34dccd4a2b7'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-23T08:37:56.631783+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19ecde-f2d5-677d-8001-e353d135e490'}}, tasks=(), interrupts=())